# Subtitles Showcase

This notebook demonstrates how to extract and process text from `.srt` subtitle files, clean the text, remove duplicates, and finally extract vocabulary tokens (verbs and other words) using the Natural Language API.

In [ ]:
%load_ext autoreload

In [ ]:
%autoreload 2
from setup_imports import *  # noqa: F401,F403


from subtitles import read_subtitles, process_subtitles, get_subtitle_tokens
from nlp import get_verbs_and_vocab
from phrases.phrase_model import get_unique_tokens_from_phrases
from phrases.search import get_phrases_by_collection, find_phrases_by_vocab_dict

## 1. Reading Subtitles
We use `pysrt` to read the subtitles from the `data/subtitles/` directory.

In [ ]:
subtitle_file = "../data/subtitles/Trouble_sv_only.srt"

try:
    subs = read_subtitles(subtitle_file)
    print(f"Successfully loaded {len(subs)} subtitle entries.")
    print("\nFirst 3 entries:")
    for i in range(min(3, len(subs))):
        print(f"{i+1}: {subs[i].text}")
except Exception as e:
    print(f"Error reading subtitles: {e}")

## 2. Processing Subtitles
We will now clean the subtitles by removing Closed Captioning (CC) info enclosed in square brackets `[like this]`, replace newlines with spaces, strip extra whitespace, and remove duplicates to get a unique list of phrases.

In [ ]:
import string

for char in "-Vi":
    print(char, char in string.punctuation)

In [ ]:
unique_phrases = process_subtitles(subs, language_code="sv", split_on_space=True)

print(f"Total unique phrases extracted: {len(unique_phrases)}")
print("\nSample of 10 unique phrases:")
for phrase in unique_phrases[:100]:
    print(f"{phrase}")

## 3. Extracting Tokens
Using `src.nlp`, we'll run a sample of these phrases through the Google Cloud Natural Language API to extract lemmas and group them into `verbs` and `vocab`.

In [ ]:
# To avoid long API wait times in this showcase, we'll process just the first 20 phrases.
sample_phrases = unique_phrases[:20]

print(f"Extracting vocabulary from {len(unique_phrases)} phrases...")
vocab_dict = get_verbs_and_vocab(unique_phrases, language="sv-se")

print("\n--- Extracted Verbs ---")
print(vocab_dict.get("verbs", []))

print("\n--- Extracted Vocab (Non-Verbs) ---")
print(vocab_dict.get("vocab", []))

In [ ]:
p, m = find_phrases_by_vocab_dict(vocab_dict, language="sv-SE")

In [ ]:
from subtitles import filter_wiktionary_words

missing_verbs = m["verbs"]
missing_valid_verbs = filter_wiktionary_words(missing_verbs, "sv")
missing_vocab = m["vocab"]
missing_valid_vocab = filter_wiktionary_words(missing_vocab, "sv")

In [ ]:
vocab_dict_save = {"verbs": missing_valid_verbs, "vocab": missing_valid_vocab}

In [ ]:
from utils import save_json

save_json(vocab_dict_save, file_path="../data/total_vocab_dict.json")

In [ ]:
[phrase.translations["sv-SE"].text for phrase in p]

In [ ]:
film_tokens = get_subtitle_tokens(unique_phrases, "sv")

In [ ]:
sorted(film_tokens)

In [ ]:
lm1000 = get_phrases_by_collection("LM1000")

In [ ]:
lm1000_str = [(x._get_translation("sv-SE").text) for x in lm1000[:10]]

In [ ]:
results = find_phrases_by_vocab_dict(
    vocab_dict={"verbs": ["springa", "äta"], "vocab": ["hund", "bil"]},
    language="sv-SE",
    collection="LM1000",
)
for p in results:
    print(p.english, p.translations["sv-SE"].text)

In [ ]:
get_

In [ ]:
lm1000_tokens = get_unique_tokens_from_phrases(lm1000, "sv-SE")
lm1000_tokens = list(map(lambda x: x.lower(), lm1000_tokens))

In [ ]:
len(film_tokens)

In [ ]:
len(lm1000_tokens)

In [ ]:
(set(map(lambda x: x[:-1], film_tokens)).difference(set(lm1000_tokens)))

In [ ]:
film_tokens